# CLIP with swappable weights

Ramen adapts only the **normalization affine parameters** (LayerNorm weight/bias for ViTs, BatchNorm for ResNets) of CLIP's
image encoder. Here those parameters are a plain dict `weights`, so you can evaluate CLIP under any set of them:

- `clip(weights, images)` → logits (full image-encoder forward with `weights`)
- `clip(weights, embeddings)` → logits from already-computed image embeddings
- `run_inference(domain, weights)` → embeddings, logits, probabilities and predictions for a corruption
- `support_set_entropy(weights, support)` → entropy of the predictions on a support set (differentiable w.r.t. `weights`)

In [ ]:
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '1')  # pick a free GPU; must be set before torch is imported

import sys
from collections import OrderedDict
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.func import functional_call
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

ROOT = os.path.dirname(os.getcwd())
sys.path.insert(0, os.path.join(ROOT, 'src'))

from models.get_pretrained_model import get_pretrained_model
from models.clip_utils import encode_text
from datasets import get_dataset_class
from methods.losses import softmax_entropy

## Settings

In [ ]:
DATASET    = 'CIFAR10C'        # 'CIFAR10C' | 'CIFAR100C'
MODEL      = 'clip_vitbase32'  # 'clip_vitbase32' | 'clip_vitbase16' | 'clip_vitlarge14' | 'clip_rn50' | 'clip_rn101'
SEVERITY   = 5
DATA_ROOT  = os.path.expanduser('~/data')
BATCH_SIZE = 256
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Load model, datasets and text classifier

In [ ]:
model, preprocess = get_pretrained_model(SimpleNamespace(model=MODEL, device=DEVICE))
model.eval().requires_grad_(False)
visual = model.visual
DTYPE = visual.conv1.weight.dtype

datasets = get_dataset_class(DATASET)(root=DATA_ROOT, severity=SEVERITY, transform=preprocess)
DOMAINS = list(datasets.environments)
CLASSES = list(datasets.classes)

# Same prompt ensemble as Ramen / Tent (models/ModelForBySampleTTA.py)
TEMPLATES = ["itap of a {}.", "a bad photo of the {}.", "a origami {}.", "a photo of the large {}.",
             "a {} in a video game.", "art of the {}.", "a photo of the small {}."]
with torch.no_grad():
    TEXT_FEATURES = encode_text(model, CLASSES, TEMPLATES)  # (C, D), L2-normalised
LOGIT_SCALE = 100.0

print(f'{DATASET}: {len(DOMAINS)} corruptions x {len(datasets[0])} images, {len(CLASSES)} classes')
print('text features:', tuple(TEXT_FEATURES.shape))

## Weights

`weights` is an `OrderedDict {param_name: tensor}` over the norm-layer affine parameters of `model.visual`, in module order
(`ln_pre`, `transformer.resblocks.*.ln_1/ln_2`, `ln_post`). `flatten_weights` gives the same layout as Ramen's per-sample
gradient vector (`get_by_sample_grad`), so a Ramen gradient `g` gives adapted weights `unflatten_weights(flatten_weights(W0) - lr * g.sign())`.

In [ ]:
NORM_TYPES = (nn.LayerNorm, nn.BatchNorm2d)


def base_weights():
    """Fresh copy of the pretrained norm affine params (fp32)."""
    w = OrderedDict()
    for name, m in visual.named_modules():
        if isinstance(m, NORM_TYPES) and m.weight is not None:
            w[f'{name}.weight'] = m.weight.detach().clone().float()
            w[f'{name}.bias'] = m.bias.detach().clone().float()
    return w


def flatten_weights(weights):
    return torch.cat([v.flatten() for v in weights.values()])


def unflatten_weights(vec, like=None):
    like = like or W0
    out, i = OrderedDict(), 0
    for k, v in like.items():
        out[k] = vec[i:i + v.numel()].view_as(v)
        i += v.numel()
    assert i == vec.numel(), f'vector has {vec.numel()} entries, weights need {i}'
    return out


W0 = base_weights()
print(f'{len(W0) // 2} norm layers, {flatten_weights(W0).numel()} adaptable params')

## `clip(weights, x)`

- `x` of shape `(B, 3, H, W)` (preprocessed images): runs the image encoder with `weights` swapped in, so the result is differentiable w.r.t. them.
- `x` of shape `(B, D)` (image embeddings): only the text classifier is applied. `weights` are **not used** in this case, since they live inside the image encoder; the embeddings already reflect whatever weights produced them.

Returns logits `(B, C)`; pass `return_probs=True` for softmax probabilities.

In [ ]:
def encode_images(weights, images):
    """L2-normalised image embeddings (B, D) computed with the given norm weights (None = pretrained)."""
    images = images.to(DEVICE, DTYPE)
    if weights is None:
        feats = visual(images)
    else:
        feats = functional_call(visual, {k: v.to(DEVICE) for k, v in weights.items()}, (images,))
    return F.normalize(feats.float(), dim=-1)


def clip(weights, x, return_probs=False):
    """Prediction vector for images (B,3,H,W) or image embeddings (B,D)."""
    if x.ndim == 4:
        emb = encode_images(weights, x)
    elif x.ndim == 2:
        emb = F.normalize(x.to(DEVICE).float(), dim=-1)
    else:
        raise ValueError(f'expected images (B,3,H,W) or embeddings (B,D), got shape {tuple(x.shape)}')
    logits = LOGIT_SCALE * emb @ TEXT_FEATURES.float().T
    return logits.softmax(-1) if return_probs else logits

## Inference over a corruption

`run_inference(domain, weights=None, n=None)` returns a dict of CPU tensors, row `i` ↔ image `index[i]` of that corruption:
`emb (N,D)`, `logits (N,C)`, `probs (N,C)`, `pred (N,)`, `label (N,)`, `index (N,)`.
`load_images(domain, index)` gives back the preprocessed images for any rows (e.g. a support set).

In [ ]:
def _domain_idx(domain):
    return DOMAINS.index(domain) if isinstance(domain, str) else int(domain)


def load_images(domain, index):
    ds = datasets[_domain_idx(domain)]
    return torch.stack([ds[int(i)][0] for i in index])


@torch.no_grad()
def run_inference(domain, weights=None, n=None, seed=0, batch_size=BATCH_SIZE, num_workers=4):
    """Embeddings + predictions for one corruption. n = random subset size (None = all images)."""
    ds = datasets[_domain_idx(domain)]
    index = np.arange(len(ds)) if n is None else np.sort(np.random.default_rng(seed).choice(len(ds), n, replace=False))
    loader = DataLoader(Subset(ds, index), batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

    embs, labels = [], []
    for images, y in tqdm(loader, desc=str(DOMAINS[_domain_idx(domain)])):
        embs.append(encode_images(weights, images).cpu())
        labels.append(y)
    emb = torch.cat(embs)
    logits = clip(None, emb).cpu()
    probs = logits.softmax(-1)
    out = dict(emb=emb, logits=logits, probs=probs, pred=probs.argmax(-1), label=torch.cat(labels),
               index=torch.as_tensor(index))
    print(f'  acc = {(out["pred"] == out["label"]).float().mean() * 100:.2f}%  ({len(index)} images)')
    return out

In [ ]:
out = run_inference('gaussian_noise', n=2000)        # pretrained weights; weights=W0 gives the same result
{k: tuple(v.shape) for k, v in out.items()}

All corruptions at once (slow on the full set; use `n` for a quick look):

In [ ]:
# all_out = {d: run_inference(d, n=1000) for d in DOMAINS}

Sanity check: the three routes to a prediction agree.

In [ ]:
with torch.no_grad():
    imgs = load_images('gaussian_noise', out['index'][:8])
    a = clip(None, imgs)            # pretrained, no functional_call
    b = clip(W0, imgs)              # explicit weights via functional_call
    c = clip(W0, out['emb'][:8])    # from stored embeddings
print('max |a-b| =', (a - b).abs().max().item(), '  max |a-c| =', (a - c).abs().max().item())

## Entropy of a support set

`support_set_entropy(weights, support)` is the softmax entropy of `clip(weights, support)`; `support` can be images or embeddings.
- `reduction`: `'mean'` (default), `'sum'` (what Ramen back-propagates) or `'none'` (per-sample entropies).
- `sample_weights`: optional `(k,)` weights, e.g. Ramen's `exp(-H_j) * exp(-beta * ||z_i - z_j||)`; returns `Σ w_j H_j` (normalised to sum 1 if `normalize=True`).

With images it is differentiable w.r.t. `weights` (gradients flow through the image encoder); with embeddings it is not.

In [ ]:
def support_set_entropy(weights, support, reduction='mean', sample_weights=None, normalize=False):
    H = softmax_entropy(clip(weights, support), reduction='none')  # (k,)
    if sample_weights is not None:
        sw = sample_weights.to(H)
        if normalize:
            sw = sw / sw.sum()
        return (sw * H).sum()
    if reduction == 'none':
        return H
    return H.sum() if reduction == 'sum' else H.mean()


def nearest(query_emb, pool_emb, k, exclude_self=True):
    """Indices (into pool) and L2 distances of the k nearest pool embeddings to one query embedding."""
    d = torch.cdist(query_emb[None].float(), pool_emb.float())[0]
    if exclude_self:
        d[d < 1e-6] = float('inf')
    dist, idx = d.topk(k, largest=False)
    return idx, dist

Example: support set = 20 nearest neighbours of one query in embedding space.

In [ ]:
q, K = 0, 20
nbr, dist = nearest(out['emb'][q], out['emb'], K)

with torch.no_grad():
    print('mean entropy (embeddings):', support_set_entropy(W0, out['emb'][nbr]).item())
    support_imgs = load_images('gaussian_noise', out['index'][nbr])
    print('mean entropy (images)    :', support_set_entropy(W0, support_imgs).item())
    print('per-sample entropy       :', support_set_entropy(W0, support_imgs, reduction='none').cpu().numpy().round(3))

Optional: one Ramen-style signSGD step on the support set's entropy, then compare entropy and the query's prediction under
the adapted weights. (Ramen uses `lr = 0.01` and the entropy/distance-weighted sum of cached per-sample gradients.)

In [ ]:
LR, BETA = 0.01, 5.0

W = OrderedDict((k, v.clone().requires_grad_(True)) for k, v in W0.items())
with torch.no_grad():
    H_j = support_set_entropy(W0, support_imgs, reduction='none')
alpha = torch.exp(-H_j) * torch.exp(-BETA * dist.to(H_j))     # Ramen's aggregation weights

loss = support_set_entropy(W, support_imgs, sample_weights=alpha)
grads = torch.autograd.grad(loss, list(W.values()))
W_adapted = OrderedDict((k, (v - LR * g.sign()).detach()) for (k, v), g in zip(W.items(), grads))

query_img = load_images('gaussian_noise', out['index'][[q]])
with torch.no_grad():
    print(f'support entropy  : {support_set_entropy(W0, support_imgs).item():.4f} -> {support_set_entropy(W_adapted, support_imgs).item():.4f}')
    p0, p1 = clip(W0, query_img, return_probs=True)[0], clip(W_adapted, query_img, return_probs=True)[0]
print(f'query (true {CLASSES[out["label"][q]]}): {CLASSES[p0.argmax()]} ({p0.max():.3f}) -> {CLASSES[p1.argmax()]} ({p1.max():.3f})')